# API — Akuisisi Data Cuaca Historis via REST API Open-Meteo
**Nama:** Yuliana Farida | **NIM:** 2501010372 | **Email:** faridayuli0903@gmail.com

In [1]:
# ==============================================================================
# UTS DATA ACQUISITION & MANAGEMENT - KELAS F
# Topik : Sektor Budaya Provinsi Bali (Warisan Budaya & Museum)
# Nama  : Yuliana Farida
# NIM   : 2501010372
# Email : faridayuli0903@gmail.com
# Prodi : Informatika - INSTIKI
# ==============================================================================
import os, re, io, sys, json, time, shutil, hashlib, datetime as dt
from pathlib import Path
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser

NAMA = "Yuliana Farida"
NIM = "2501010372"
EMAIL_PRIBADI = "faridayuli0903@gmail.com"
UA = f"INSTIKI-UTS-DAM-BUDAYA/1.0 ({EMAIL_PRIBADI}; {NAMA}; NIM {NIM})"
TIMEOUT = 25
JEDA = 2.0

BASE_DIR = Path(f"UTS_DAM_BUDAYA_{NIM}")
RAW = BASE_DIR / "Data/Raw"
PROC = BASE_DIR / "Process"
DOCS = BASE_DIR / "docs"
for p in (RAW, PROC, DOCS):
    p.mkdir(parents=True, exist_ok=True)

PROV_FILE = DOCS / "provenance.csv"

def hitung_md5(filepath):
    h = hashlib.md5()
    with open(filepath, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def catat_provenance(sumber, url, filepath, lisensi, metode):
    row = {
        "sumber": sumber,
        "url": url,
        "tanggal_akses": dt.datetime.now().isoformat(timespec="seconds"),
        "lisensi": lisensi,
        "metode_akuisisi": metode,
        "nama_berkas": str(filepath),
        "ukuran_byte": Path(filepath).stat().st_size,
        "md5_hash": hitung_md5(filepath)
    }
    df_baru = pd.DataFrame([row])
    if PROV_FILE.exists():
        df_lama = pd.read_csv(PROV_FILE)
        df_baru = pd.concat([df_lama[df_lama["sumber"] != sumber], df_baru], ignore_index=True)
    df_baru.to_csv(PROV_FILE, index=False)

def request_etis(url, params=None, max_retry=4):
    for i in range(1, max_retry + 1):
        try:
            r = requests.get(url, params=params, headers={"User-Agent": UA}, timeout=TIMEOUT)
            if r.status_code == 200:
                time.sleep(JEDA)
                return r
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** i)
                continue
            r.raise_for_status()
        except (requests.Timeout, requests.ConnectionError):
            time.sleep(2 ** i)
    raise RuntimeError(f"Gagal mengambil {url}")

print("Konfigurasi Topik Budaya selesai. User-Agent:", UA)

In [1]:
# SOAL 2: Klasifikasi kolom PII / quasi-identifier dan Data Ethics Checklist 9 butir (Tanpa Wikipedia)
PII_PATTERN = r"nik|nama_pemilik|email|telp|phone|alamat_pribadi|ktp|npwp"
QUASI_PATTERN = r"lat|lon|latitude|longitude|kode_?pos|kecamatan|desa|kabupaten|kota|tahun|date"

df_pii = pd.DataFrame([
    {
        "kolom": c,
        "kategori": "PII Langsung" if re.search(PII_PATTERN, c, re.I)
                    else ("Quasi-Identifier" if re.search(QUASI_PATTERN, c, re.I) else "Non-PII")
    }
    for c in df_raw.columns
])
df_pii.to_csv(DOCS / "klasifikasi_pii.csv", index=False)

checklist_9 = pd.DataFrame({
    "no": range(1, 10),
    "butir_etika": [
        "1. Sumber data dan URL dicantumkan",
        "2. Lisensi data diperiksa dan dicatat",
        "3. robots.txt / Terms of Service diperiksa",
        "4. Kolom PII / quasi-identifier diidentifikasi",
        "5. PII dihapus, di-mask, atau di-pseudonimisasi",
        "6. Prinsip minimalisasi data diterapkan",
        "7. Potensi bias sampel dianalisis",
        "8. Tujuan penggunaan data dinyatakan eksplisit",
        "9. Rencana retensi dan penghapusan data ada"
    ],
    "status": ["Ya"] * 9,
    "bukti_implementasi": [
        "Tercatat lengkap pada docs/provenance.csv beserta hash MD5",
        "Lisensi Open Data Publik, CC BY 4.0 (Open-Meteo), dan Data Publik Resmi Instansi Pemerintah",
        "Bukti pengecekan robots.txt tersimpan pada docs/robots_check.json",
        "Hasil klasifikasi setiap kolom tersimpan pada docs/klasifikasi_pii.csv",
        "Kolom alamat/koordinat rinci dihapus sebelum proses penggabungan data",
        "Hanya mengambil kolom wilayah dan agregasi jumlah objek budaya",
        "Bias pencatatan pada situs budaya/museum yang terdaftar resmi",
        "Digunakan murni untuk tugas UTS sektor Budaya (non-komersial)",
        "Data mentah dihapus dari penyimpanan lokal setelah evaluasi semester selesai"
    ]
})
checklist_9.to_csv(DOCS / "data_ethics_checklist.csv", index=False)
display(checklist_9)